# Abstractive Summarization on News with T5

Runs base and fine‑tuned t5-small on CNN/DailyMail, generates summaries for thousands of articles, and evaluates with ROUGE, BERTScore, and GPT‑2 perplexity, including qualitative article‑level comparisons

In [1]:
!pip install transformers[torch] datasets evaluate rouge_score pandas
!pip install accelerate bitsandbytes
!pip install bert_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.6 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=1525a09b49f1ca6295eba697a0f67c1b4b7977b0d5d4b26e74ae3fa793314a34
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.1 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import torch
import evaluate
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from google.colab import files
from tqdm import tqdm

print("Please upload your 'cnn_dailymail_5percent.csv' file.")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

df_part1 = pd.read_csv(file_name)
hf_dataset = Dataset.from_pandas(df_part1)

# You can also use this to see your data
print(hf_dataset)





Please upload your 'cnn_dailymail_5percent.csv' file.


Saving cnn_dailymail_5percent.csv to cnn_dailymail_5percent.csv
Dataset({
    features: ['article', 'highlights', 'id'],
    num_rows: 15599
})


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model_names = {
    "t5_small": "google-t5/t5-small",
    "t5_finetuned": "ubikpt/t5-small-finetuned-cnn"
}

models = {}
tokenizers = {}

print("Loading models and tokenizers...")
for name, path in model_names.items():
    tokenizers[name] = AutoTokenizer.from_pretrained(path)
    models[name] = AutoModelForSeq2SeqLM.from_pretrained(path).to(device)
print("Models loaded.")



Loading models and tokenizers...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/242M [00:00<?, ?B/s]

Models loaded.


In [ ]:
prefix = "summarize: "

def preprocess_function_part1(examples):
    inputs = [prefix + doc for doc in examples["article"]]
    model_inputs = tokenizers["t5_small"](inputs, max_length=1024, padding=True, truncation=True)
    return model_inputs

tokenized_dataset = hf_dataset.map(preprocess_function_part1, batched=True)
tokenized_dataset.set_format("torch", columns=["input_ids", "attention_mask"])
references = hf_dataset['highlights']



Map:   0%|          | 0/15599 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

In [ ]:
all_predictions = {}


def custom_collate_fn(batch):
    input_ids = [item['input_ids'] for item in batch]
    attention_mask = [item['attention_mask'] for item in batch]

    max_len = max([len(ids) for ids in input_ids])
    padded_input_ids = torch.stack([torch.nn.functional.pad(ids, (0, max_len - len(ids)), value=tokenizers["t5_small"].pad_token_id) for ids in input_ids])
    padded_attention_mask = torch.stack([torch.nn.functional.pad(mask, (0, max_len - len(mask)), value=0) for mask in attention_mask])

    return {'input_ids': padded_input_ids, 'attention_mask': padded_attention_mask}


for name, model in models.items():
    print(f"Generating summaries for: {name}")
    predictions = []
    dataloader = torch.utils.data.DataLoader(tokenized_dataset, batch_size=16, collate_fn=custom_collate_fn)

    for batch in tqdm(dataloader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=150,
            num_beams=4,
            early_stopping=True
        )


        decoded_preds = tokenizers[name].batch_decode(outputs, skip_special_tokens=True)
        predictions.extend(decoded_preds)

    all_predictions[name] = predictions

print("Inference complete.")

Generating summaries for: t5_small


100%|██████████| 975/975 [23:18<00:00,  1.43s/it]


Generating summaries for: t5_finetuned


100%|██████████| 975/975 [12:04<00:00,  1.35it/s]

Inference complete.


In [ ]:
rouge = evaluate.load('rouge')
bertscore = evaluate.load("bertscore")
perplexity = evaluate.load("perplexity", module_type="metric")

print("--- Evaluation Results ---")

for name, predictions in all_predictions.items():
    print(f"\nResults for: {name}")

    # ROUGE
    rouge_results = rouge.compute(predictions=predictions, references=references)
    print(f"ROUGE-1: {rouge_results['rouge1']}")
    print(f"ROUGE-2: {rouge_results['rouge2']}")

    # BERTSCORE
    print("Calculating BERTSCORE (this may take a while)...")
    bert_results = bertscore.compute(predictions=predictions, references=references, lang="en")
    # We take the mean across all samples
    print(f"BERTSCORE-Precision: {sum(bert_results['precision']) / len(bert_results['precision'])}")
    print(f"BERTSCORE-Recall: {sum(bert_results['recall']) / len(bert_results['recall'])}")
    print(f"BERTSCORE-F1: {sum(bert_results['f1']) / len(bert_results['f1'])}")

    # Perplexity
    print("Calculating Perplexity (using 'gpt2' as per instructions)...")
    perp_results = perplexity.compute(model_id='gpt2', predictions=predictions)
    print(f"Perplexity (mean): {perp_results['mean_perplexity']}")


print("\n--- Qualitative Comparison ---")
article_indices = [10, 20]

for index in article_indices:
    print(f"\n--- ARTICLE {index} ---")
    original_article = hf_dataset[index]['article']
    reference_summary = hf_dataset[index]['highlights']

    print(f"**Original Article (snippet):**\n{original_article[:500]}...\n")
    print(f"**Reference Summary:**\n{reference_summary}\n")

    for name, predictions in all_predictions.items():
        generated_summary = predictions[index]
        print(f"**Summary from {name}:**\n{generated_summary}\n")

    print("-" * 30)

--- Evaluation Results ---

Results for: t5_small
ROUGE-1: 0.342744017508306
ROUGE-2: 0.14971441063343532
Calculating BERTSCORE (this may take a while)...


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

  0%|          | 0/975 [00:00<?, ?it/s]

Perplexity (mean): 54.49249871330878

Results for: t5_finetuned
ROUGE-1: 0.287970211439898
ROUGE-2: 0.13466519763524795
Calculating BERTSCORE (this may take a while)...
BERTSCORE-Precision: 0.8871179343767751
BERTSCORE-Recall: 0.8520747808640076
BERTSCORE-F1: 0.8690466276765886
Calculating Perplexity (using 'gpt2' as per instructions)...


  0%|          | 0/975 [00:00<?, ?it/s]

Perplexity (mean): 102.18842301535923

--- Qualitative Comparison ---

--- ARTICLE 10 ---
**Original Article (snippet):**
Well, that was quick. Just hours after going on sale in the U.S., Canada and the UK, the OUYA gaming console was already sold out Tuesday morning on Amazon, though other retailers still had it in stock. Amazon, which was selling the device for $99, told customers that the item was temporarily out of stock. However, as of Tuesday morning, Target and Best Buy were still carrying OUYA. GameStop noted that the item was "currently unavailable." SEE ALSO: 7 Gadgets for the Ultimate Connected Living Ro...

**Reference Summary:**
OUYA gaming console sells out in hours on Amazon .
Android-based console sells for $99 .

**Summary from t5_small:**
the OUYA gaming console went on sale in the united states, canada and the uk. the item was temporarily out of stock, but other retailers still carry it. OUYA launched on Kickstarter as an open gaming console.

**Summary from t5_finet